# PEDE - PDF to Model Embedding (Google Colab)
Pipeline ini mengubah PDF menjadi format Markdown, memecahnya menjadi *chunks*, membuat *vector embeddings* dengan BAAI/bge-m3, dan memasukkannya ke Qdrant Database.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Setup Kode (Clone / Git Pull)
Clone repositori jika belum ada di Colab, atau jalankan `git pull` untuk mengambil update kode terbaru.

In [ ]:
import os

REPO_URL = "https://github.com/ifcoid/PEDE.git"
WORK_DIR = "/content/PEDE"

if not os.path.exists(WORK_DIR):
    print(f"Repositori belum ada. Cloning ke {WORK_DIR} ...")
    !git clone {REPO_URL} {WORK_DIR}
else:
    print("Repositori sudah ada. Menjalankan git pull untuk update kode ...")
    !git -C {WORK_DIR} pull

os.chdir(WORK_DIR)
print("Current Working Directory:", os.getcwd())

## 2. Instalasi Dependensi

In [ ]:
!pip install -r requirements.txt
!pip install FlagEmbedding qdrant-client sentence-transformers PyMuPDF httpx python-dotenv paddleocr paddlepaddle-gpu<3.0.0
!apt-get install -y poppler-utils

## 3. Konfigurasi (Qdrant & Notifikasi)
Semua kredensial diambil dari **🔑 Secrets** Colab (sidebar kiri). Tambahkan secret berikut lalu aktifkan *Notebook access*:
- `QDRANT_URL` — endpoint Qdrant Cloud
- `QDRANT_API_KEY` — API key Qdrant
- `TELEGRAM_BOT_TOKEN` *(opsional)* — token bot dari [@BotFather](https://t.me/BotFather)
- `TELEGRAM_CHAT_ID` *(opsional)* — chat ID tujuan notifikasi

Folder PDF di-set langsung lewat variabel `PDF_FOLDER` di cell bawah (ubah sesuai lokasi di Google Drive Anda).

In [ ]:
import os
from google.colab import userdata

# === Path folder PDF di Google Drive (ubah sesuai lokasi Anda) ===
PDF_FOLDER = "/content/drive/MyDrive/pdfs"


def _secret(name: str) -> str:
    """Ambil nilai secret dari Colab; kembalikan string kosong bila belum diset."""
    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


# === Kredensial dari Colab Secrets ===
os.environ["QDRANT_URL"] = _secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = _secret("QDRANT_API_KEY")
TG_TOKEN = _secret("TELEGRAM_BOT_TOKEN")
TG_CHAT_ID = _secret("TELEGRAM_CHAT_ID")

assert os.environ["QDRANT_URL"] and os.environ["QDRANT_API_KEY"], \
    "Secret QDRANT_URL / QDRANT_API_KEY belum diset di panel 🔑 Secrets!"

print("Konfigurasi OK | Folder PDF:", PDF_FOLDER)
print("Telegram:", "aktif" if (TG_TOKEN and TG_CHAT_ID) else "nonaktif (secret belum diset)")

## 4. Eksekusi Proses Ingestion
Skrip ini memproses semua PDF di folder, mengirim *vector embeddings* ke Qdrant, lalu mengirim notifikasi Telegram (jika diisi).

**Auto-retry:** selama kernel Colab masih hidup, proses otomatis diulang (maks. `MAX_ATTEMPTS`, jeda `RETRY_DELAY` detik) bila ada PDF yang gagal — berguna untuk gangguan internet sesaat. PDF yang sudah masuk akan di-skip otomatis, jadi tiap putaran hanya mengejar yang belum selesai.

In [ ]:
import os, re, time, glob, subprocess
import requests

folder_path = PDF_FOLDER
TG_API = f"https://api.telegram.org/bot{TG_TOKEN}"
TG_ON = bool(TG_TOKEN and TG_CHAT_ID)

# === Auto-retry (selama kernel masih hidup) ===
MAX_ATTEMPTS = 4    # total percobaan (1 awal + 3 retry)
RETRY_DELAY = 30    # jeda antar percobaan (detik)


def tg_send(text: str):
    """Kirim pesan baru; kembalikan message_id (atau None)."""
    if not TG_ON:
        return None
    try:
        r = requests.post(f"{TG_API}/sendMessage",
                          data={"chat_id": TG_CHAT_ID, "text": text[:4000], "parse_mode": "HTML"},
                          timeout=30)
        if r.status_code == 200:
            return r.json().get("result", {}).get("message_id")
        print(f"Gagal kirim Telegram: {r.status_code} - {r.text}")
    except Exception as e:
        print(f"Error kirim Telegram: {e}")
    return None


def tg_edit(message_id, text: str):
    """Update isi pesan progress (tidak memicu notifikasi suara)."""
    if not (TG_ON and message_id):
        return
    try:
        requests.post(f"{TG_API}/editMessageText",
                      data={"chat_id": TG_CHAT_ID, "message_id": message_id,
                            "text": text[:4000], "parse_mode": "HTML"},
                      timeout=30)
    except Exception as e:
        print(f"Error edit Telegram: {e}")


def grab(text: str, pattern: str, default="?"):
    mt = re.search(pattern, text)
    return mt.group(1).strip() if mt else default


n_pdf = len(glob.glob(os.path.join(folder_path, "**", "*.pdf"), recursive=True))
print(f"Memproses {n_pdf} PDF di dalam: {folder_path}")
print("=========================================")
if not TG_ON:
    print("Telegram nonaktif (secret belum diset) - notifikasi dilewati.")

# Pesan progress yang di-update live; bila kernel mati, posisi terakhir tetap terlihat.
progress_mid = tg_send(
    f"🚀 <b>PEDE Ingestion dimulai</b>\n"
    f"Folder: <code>{folder_path}</code>\n"
    f"Total PDF: {n_pdf}"
)

re_proc = re.compile(r"Processing:\s*(.+?)\s*$")
overall_start = time.time()


def run_once(attempt: int):
    """Jalankan ingest.py sekali: streaming log + update progress. Return (returncode, out, error)."""
    captured, done, err, rc = [], 0, "", -1
    try:
        proc = subprocess.Popen(
            ["python", "ingest.py", folder_path],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")              # streaming live
            captured.append(line)
            m = re_proc.search(line)
            if m:                            # tanda mulai (atau skip) satu PDF
                done += 1
                cur = m.group(1)
                tag = f" (percobaan {attempt}/{MAX_ATTEMPTS})" if attempt > 1 else ""
                tg_edit(progress_mid,
                        f"⏳ <b>PEDE Ingestion berjalan</b>{tag}\n"
                        f"Folder: <code>{folder_path}</code>\n"
                        f"Progres: {done}/{n_pdf}\n"
                        f"Sedang diproses: <code>{cur}</code>\n"
                        f"Elapsed total: {(time.time()-overall_start)/60:.1f} menit")
        rc = proc.wait()
    except Exception as e:
        err, rc = str(e), -1
    return rc, "".join(captured), err


# === Loop: ulang sampai bersih (Failed=0) atau jatah percobaan habis ===
returncode, out, error_msg, attempt = -1, "", "", 0
while attempt < MAX_ATTEMPTS:
    attempt += 1
    if attempt > 1:
        msg = f"🔁 Percobaan {attempt}/{MAX_ATTEMPTS} - PDF yang sudah masuk akan di-skip..."
        print(f"\n{msg}")
        tg_edit(progress_mid, msg)
    returncode, out, error_msg = run_once(attempt)
    failed = grab(out, r"Failed:\s*(\d+)")
    if returncode == 0 and failed == "0":
        break  # semua PDF sukses
    if attempt < MAX_ATTEMPTS:
        print(f"\nMasih ada kegagalan (Failed={failed}, exit={returncode}). "
              f"Mengulang dalam {RETRY_DELAY}s...")
        time.sleep(RETRY_DELAY)

elapsed = time.time() - overall_start
processed = grab(out, r"Processed:\s*(\d+)")
success   = grab(out, r"Success:\s*(\d+)")
failed    = grab(out, r"Failed:\s*(\d+)")
total     = grab(out, r"Collection total:\s*(\d+)")

if returncode == 0 and failed == "0":
    status = "✅ SUKSES"
else:
    status = f"⚠️ SELESAI dengan kegagalan (Failed={failed}, exit={returncode})"

summary = (
    f"<b>PEDE Ingestion Selesai</b>\n"
    f"Status: {status}\n"
    f"Percobaan: {attempt}/{MAX_ATTEMPTS}\n"
    f"Folder: <code>{folder_path}</code>\n"
    f"Durasi total: {elapsed/60:.1f} menit\n\n"
    f"<b>Ringkasan:</b>\n"
    f"PDF ditemukan: {n_pdf}\n"
    f"Processed: {processed} | Success: {success} | Failed: {failed}\n"
    f"Total chunk di Qdrant: {total}"
)
if error_msg:
    summary += f"\nError: <code>{error_msg}</code>"

print("\n=========================================")
print(f"Selesai dalam {elapsed/60:.1f} menit | {status}")
tg_edit(progress_mid, summary)   # update pesan progress ke status final
tg_send(summary)                 # pesan final (memicu notifikasi)